# Cloud Optimizer - Experiment Analysis
This notebook analyzes the results from the metaheuristic scheduling algorithms.
We have two main datasets:
1. `experiment_results.csv`: Contains results of various algorithms across different population sizes, iterations, and diversity settings.
2. `hyperparam_results.csv`: Contains sensitivity analysis of algorithm-specific hyperparameters.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load data
df_exp = pd.read_csv('../experiment_results.csv')
df_hyp = pd.read_csv('../hyperparam_results.csv')

display(df_exp.head())
display(df_hyp.head())

,Algorithm,Seed,Pop_Size,Iterations,Diversity_Freq,Makespan,Cost,Fitness
0,GA,42,20,50,0,1172.87,291.97,2046.31
1,PSO,42,20,50,0,1205.65,287.18,2038.71
2,GWO,42,20,50,0,1912.04,138.59,1648.96
3,BCO,42,20,50,0,1248.84,289.28,2070.83
4,WOA,42,20,50,0,1417.83,289.95,2158.68


,Algorithm,Seed,Parameter_Combo,Makespan,Cost,Fitness
0,GA,42,"Pc=0.7, Pm=0.01",1247.79,273.65,1992.15
1,GA,42,"Pc=0.7, Pm=0.05",1180.05,294.07,2060.36
2,GA,42,"Pc=0.7, Pm=0.1",1196.93,290.37,2050.34
3,GA,42,"Pc=0.85, Pm=0.01",1164.69,299.85,2081.57
4,GA,42,"Pc=0.85, Pm=0.05",1199.48,291.56,2057.54


## 1. Overall Algorithm Performance
Let's compare the average Makespan, Cost, and Fitness across all algorithms.

In [2]:
# Group by Algorithm
algo_perf = df_exp.groupby('Algorithm', as_index=False)[['Makespan', 'Cost', 'Fitness']].mean()
algo_perf = algo_perf.sort_values('Fitness')

# Build 3 Plotly bar charts in one row
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Average Fitness (Lower is Better)', 'Average Makespan', 'Average Cost')
)

fig.add_trace(
    go.Bar(x=algo_perf['Algorithm'], y=algo_perf['Fitness'], name='Fitness', marker_color='#636EFA'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=algo_perf['Algorithm'], y=algo_perf['Makespan'], name='Makespan', marker_color='#00CC96'),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=algo_perf['Algorithm'], y=algo_perf['Cost'], name='Cost', marker_color='#EF553B'),
    row=1, col=3
)

fig.update_layout(
    title='Overall Algorithm Performance',
    showlegend=False,
    height=500,
    width=1200
)
fig.update_xaxes(title_text='Algorithm', row=1, col=1)
fig.update_xaxes(title_text='Algorithm', row=1, col=2)
fig.update_xaxes(title_text='Algorithm', row=1, col=3)
fig.update_yaxes(title_text='Fitness', row=1, col=1)
fig.update_yaxes(title_text='Makespan', row=1, col=2)
fig.update_yaxes(title_text='Cost', row=1, col=3)

fig.show()

## 2. Impact of Diversity Extension
How does the `Diversity_Freq` parameter affect fitness?

In [3]:
# Keep only algorithms that actually use diversity
diversity_data = df_exp.loc[~df_exp['Algorithm'].isin(['RR', 'SJF'])].copy()
diversity_data['Diversity_Freq'] = pd.to_numeric(diversity_data['Diversity_Freq'], errors='coerce')
diversity_data = diversity_data.dropna(subset=['Diversity_Freq'])

# Build a clear summary table: mean fitness, spread, and number of runs
diversity_summary = (
    diversity_data
    .groupby(['Algorithm', 'Diversity_Freq'], as_index=False)
    .agg(
        avg_fitness=('Fitness', 'mean'),
        std_fitness=('Fitness', 'std'),
        runs=('Fitness', 'size')
    )
    .sort_values(['Algorithm', 'Diversity_Freq'])
)

print('Diversity impact summary (lower avg_fitness is better):')
display(diversity_summary)

# For each algorithm, identify the best diversity frequency
best_diversity = (
    diversity_summary.loc[
        diversity_summary.groupby('Algorithm')['avg_fitness'].idxmin(),
        ['Algorithm', 'Diversity_Freq', 'avg_fitness', 'std_fitness', 'runs']
    ]
    .sort_values('avg_fitness')
    .rename(columns={
        'Diversity_Freq': 'best_diversity_freq',
        'avg_fitness': 'best_avg_fitness',
        'std_fitness': 'fitness_std_at_best'
    })
)

print('Best diversity frequency per algorithm:')
display(best_diversity)

# Simple global recommendation: which diversity frequency is best on average
global_diversity_rank = (
    diversity_summary
    .groupby('Diversity_Freq', as_index=False)['avg_fitness']
    .mean()
    .sort_values('avg_fitness')
    .rename(columns={'avg_fitness': 'mean_fitness_across_algorithms'})
)

print('Global ranking of diversity frequency (all algorithms combined):')
display(global_diversity_rank)

best_global_freq = global_diversity_rank.iloc[0]['Diversity_Freq']
best_global_score = global_diversity_rank.iloc[0]['mean_fitness_across_algorithms']
print(f"Recommended default Diversity_Freq = {best_global_freq} (mean fitness: {best_global_score:.3f})")

Diversity impact summary (lower avg_fitness is better):


,Algorithm,Diversity_Freq,avg_fitness,std_fitness,runs
0,BCO,0,1994.365000,72.194757,12
1,BCO,10,1981.833750,51.294557,8
2,BCO,20,2066.640000,12.108361,4
3,BCO,50,1981.833750,51.294557,8
4,GA,0,2038.437500,11.937991,12
5,GA,10,2030.305556,14.620363,9
6,GA,20,2050.030000,10.154989,4
7,GA,50,2034.828750,16.652683,8
8,GWO,0,1608.608333,27.379948,12
9,GWO,10,1600.572222,30.873327,9


Best diversity frequency per algorithm:


,Algorithm,best_diversity_freq,best_avg_fitness,fitness_std_at_best,runs
11,GWO,50,1598.567500,23.284221,8
13,PSO,10,1948.556667,22.999465,9
1,BCO,10,1981.833750,51.294557,8
5,GA,10,2030.305556,14.620363,9
17,WOA,10,2099.453750,24.210174,8


Global ranking of diversity frequency (all algorithms combined):


,Diversity_Freq,mean_fitness_across_algorithms
1,10,1932.144389
3,50,1935.114250
0,0,1944.862333
2,20,1980.558500


Recommended default Diversity_Freq = 10.0 (mean fitness: 1932.144)


## 3. Hyperparameter Sensitivity
Let's look at the best hyperparameters found for each algorithm.

In [4]:
# Find best hyperparameters for each algorithm
best_hyps = df_hyp.loc[df_hyp.groupby('Algorithm')['Fitness'].idxmin()]
display(best_hyps[['Algorithm', 'Parameter_Combo', 'Fitness', 'Makespan', 'Cost']].sort_values('Fitness'))

# Plotly version: Fitness spread by hyperparameter combination
fig = px.strip(
    df_hyp,
    x='Algorithm',
    y='Fitness',
    color='Parameter_Combo',
    stripmode='group',
    title='Fitness Variance by Hyperparameter Combinations'
 )

fig.update_layout(
    width=1200,
    height=550,
    legend_title='Parameter Combo'
 )

fig.show()

,Algorithm,Parameter_Combo,Fitness,Makespan,Cost
31,PSO,"w=0.7, c1=2.0, c2=1.0",1911.20,1364.14,245.83
38,BCO,NC=10,1931.77,1308.58,255.50
0,GA,"Pc=0.7, Pm=0.01",1992.15,1247.79,273.65
46,WOA,b=1.0,2124.66,1497.70,275.16
